# 🦺 YOLOv8 EHS Training — Casco & Chaleco
**Ejecutá cada celda en orden. GPU T4 recomendada: Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Verificar GPU
!nvidia-smi
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# 2. Instalar dependencias
!pip install ultralytics roboflow -q

In [ ]:
# 3. Descargar dataset de Roboflow (PPE Detection - casco + chaleco)
from roboflow import Roboflow

# Dataset público — no necesita API key
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # crear cuenta gratis en roboflow.com
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(1)
dataset = version.download("yolov8")
print(f'Dataset descargado en: {dataset.location}')

In [ ]:
# 4. Ver clases del dataset
import yaml, os
yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(yaml_path) as f:
    data = yaml.safe_load(f)
print('Clases:', data['names'])
print('Train images:', data.get('nc'), 'clases')

In [ ]:
# 5. ENTRENAR
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano = más rápido, suficiente para POC

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='ehs-ppe-v1',
    patience=10,       # early stopping
    device=0,          # GPU
    exist_ok=True
)
print('Entrenamiento completo!')

In [ ]:
# 6. Ver métricas
from ultralytics import YOLO
model = YOLO('runs/detect/ehs-ppe-v1/weights/best.pt')
metrics = model.val()
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# 7. Probar con imagen de prueba
model = YOLO('runs/detect/ehs-ppe-v1/weights/best.pt')
results = model.predict('https://images.unsplash.com/photo-1504307651254-35680f356dfd?w=640', save=True)
from IPython.display import Image
Image('runs/detect/predict/image0.jpg')

In [ ]:
# 8. Descargar el modelo entrenado
from google.colab import files
files.download('runs/detect/ehs-ppe-v1/weights/best.pt')
print('Guardá este archivo como backend/models/ehs-ppe-v1.pt')

## ✅ Siguiente paso
1. Descargá `best.pt` y copialo a `cv-ehs-poc/backend/models/ehs-ppe-v1.pt`
2. En `detector.py` cambiá:
```python
self._model = YOLO('yolov8n.pt')
# por:
self._model = YOLO('models/ehs-ppe-v1.pt')
```